# 🚀 SupremeAI Shadow Node (Kaggle)
Run this notebook in a Kaggle environment to create a free, zero-cost shadow API node. SupremeAI uses this node to handle requests when the primary Render server is asleep or rate-limited.

## Instructions:
1. Replace `YOUR_GROQ_API_KEY_HERE` with your Groq API key.
2. Replace `YOUR_NGROK_AUTHTOKEN_HERE` with your free ngrok authtoken.
3. Click **Run All**.
4. Copy the generated `SHADOW NODE URL` and add it to your SupremeAI backend `.env` as `SHADOW_NODE_URL`.

In [ ]:
!pip install fastapi uvicorn groq pyngrok nest-asyncio

In [ ]:
import nest_asyncio
nest_asyncio.apply()
from fastapi import FastAPI, Request
import uvicorn
from groq import Groq
from pyngrok import ngrok
import os

GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
app = FastAPI()
client = Groq(api_key=GROQ_API_KEY)

@app.post("/v1/chat/completions")
async def chat(request: Request):
    data = await request.json()
    messages = data.get("messages", [])
    model = data.get("model", "llama-3.3-70b-versatile")
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages
        )
        return response.model_dump()
    except Exception as e:
        return {"error": str(e)}

@app.get("/health")
def health():
    return {"status": "alive", "type": "kaggle_shadow_node"}

# Expose port via ngrok
public_url = ngrok.connect(8000)
print("\n" + "="*60)
print(f"🔥 SHADOW NODE ACTIVE: {public_url.public_url}")
print("Set this URL in your SupremeAI .env as SHADOW_NODE_URL")
print("="*60 + "\n")

# Start Server
uvicorn.run(app, host="0.0.0.0", port=8000)
